# Practice #1. "Trend analysis"

Smooth a series, then pull its trend out.

Fill in the cells tagged `graded`. They are marked and graded automatically, so
keep every function name and signature exactly as given. Untagged cells are
yours to change.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose

# DATA_DIR is injected when the grader imports this notebook; when you run the
# notebook yourself it falls back to ../data relative to this file.
def find_data_dir():
    """The repo's data/ directory, wherever the kernel happens to start."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "data" / "airline-passengers.csv").exists():
            return folder / "data"
    raise FileNotFoundError("data/ not found - run this notebook inside the repo")


DATA_DIR = globals().get("DATA_DIR", find_data_dir())

## 0. Data reading and visualization

`opsd_germany_daily.csv` has four columns. `Wind` and `Solar` are empty
before 2012, so a bare `df.dropna()` also drops six years of good `Consumption`.
Select your two columns first, then drop.

In [ ]:
def load_series(path, time_col, value_col):
    """Read a CSV into a float Series named "y" on a DatetimeIndex."""
    # TODO: keep [time_col, value_col] -> rename to ds/y -> parse ds as datetime
    # -> dropna -> index by ds -> return y as float.
    # Select the columns before dropping, not after.
    raise NotImplementedError

In [ ]:
series = load_series(DATA_DIR / "opsd_germany_daily.csv", "Date", "Consumption")
print(f"{len(series)} points from {series.index.min():%Y-%m-%d} "
      f"to {series.index.max():%Y-%m-%d}")
series.head()

In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(series)
plt.ylabel("y")
plt.xlabel("ds")
plt.title("Germany daily electricity consumption");

## 1. Timeseries smoothing

### 1.1 Moving average

#### Centered moving average
$$x[t] = \frac{x[t - \lfloor n/2 \rfloor] + \ldots + x[t] + \ldots + x[t + \lfloor n/2 \rfloor]}{n}$$

#### Trailing moving average
$$x[t] = \frac{x[t - n + 1] + \ldots + x[t - 1] + x[t]}{n}$$

where $n$ is the window size. Only the trailing average is usable for
forecasting: the centred one reads values from the future.

In [ ]:
def centered_moving_average(series, window):
    """Window centred on each point."""
    # TODO: one Series.rolling(...).mean() call
    raise NotImplementedError


def trailing_moving_average(series, window):
    """Window ending at each point."""
    # TODO: one Series.rolling(...).mean() call
    raise NotImplementedError

In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(series, linewidth=1, alpha=0.4, label="base")
plt.plot(centered_moving_average(series, 30), label="centered MA (30)")
plt.plot(trailing_moving_average(series, 30), label="trailing MA (30)")
plt.legend();

Smooth the data with several window sizes and compare.

**Question.** How does the window size change the smoothed series, and why is
the trailing average shifted to the right relative to the centred one?

In [ ]:
# your code here — free exploration, not graded

### 1.2 Exponential smoothing

The two libraries agree to floating-point precision — but only with the
right arguments.

- `ewm(alpha=a)` defaults to `adjust=True`: a renormalised weighted average, not
  the textbook recursion. It does **not** match statsmodels.
- `adjust=False` gives exactly `s[0] = x[0]`, `s[t] = a·x[t] + (1-a)·s[t-1]`.

Implement both. The assertion below must pass.

In [ ]:
def exponential_smoothing(series, alpha):
    """SES via pandas: s[0] = x[0], s[t] = a*x[t] + (1-a)*s[t-1]."""
    # TODO: Series.ewm — check what `adjust` does before you pick its value.
    raise NotImplementedError


def exponential_smoothing_statsmodels(series, alpha):
    """Same recursion via statsmodels; return the level as a Series."""
    # TODO: SimpleExpSmoothing(initialization_method="known",
    # initial_level=series.iloc[0]).fit(smoothing_level=alpha, optimized=False)
    # Without those arguments statsmodels fits its own alpha and ignores yours.
    raise NotImplementedError

In [ ]:
pandas_smoothed = exponential_smoothing(series, 0.3)
statsm_smoothed = exponential_smoothing_statsmodels(series, 0.3)
largest_gap = (pandas_smoothed - statsm_smoothed).abs().max()
print(f"max |pandas - statsmodels| = {largest_gap:.3e}")
assert largest_gap < 1e-8, "the two implementations must agree"

In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(series, linewidth=1, alpha=0.3, label="base")
for a in (0.05, 0.3, 0.9):
    plt.plot(exponential_smoothing(series, a), label=f"alpha = {a}")
plt.legend();

**Question.** What does alpha do to the smoothed series, and what
happens in the limits alpha -> 0 and alpha -> 1?

### 1.3 Double exponential smoothing

Holt's method adds a trend component: a level `l[t]` smoothed by alpha
and a slope `b[t]` smoothed by beta.

In [ ]:
def double_exponential_smoothing(series, alpha, beta, trend="add"):
    """Holt's linear trend smoothing; return the fitted level as a Series."""
    # TODO: ExponentialSmoothing(trend=trend, initialization_method="estimated").
    # The keyword for beta is `smoothing_trend`.
    raise NotImplementedError

In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(series, linewidth=1, alpha=0.3, label="base")
plt.plot(double_exponential_smoothing(series, 0.3, 0.1), label="Holt level")
plt.legend();

Try several alpha/beta pairs and both trend types (`"add"`, `"mul"`).

**Questions.** What does beta control? Are alpha and beta related?

statsmodels may raise `ConvergenceWarning` here. Don't silence it — it means the
fit never settled, and the numbers below it are not trustworthy.

In [ ]:
# your code here — free exploration, not graded

## 2. Trend extraction

### 2.1 Linear regression

In [ ]:
def linear_trend(series):
    """Least-squares straight line through the series, as a Series."""
    # TODO: features are 0, 1, 2, ... len(series) - 1 reshaped to a column.
    raise NotImplementedError


def decomposition_trend(series, period, model="additive"):
    """Trend component from seasonal_decompose.

    Keep the NaNs at both ends: that is where a centred window does not fit.
    """
    # TODO
    raise NotImplementedError

In [ ]:
trend_line = linear_trend(series)
trend_decomposed = decomposition_trend(series, period=365)

plt.figure(figsize=(20, 5))
plt.plot(series, linewidth=1, alpha=0.3, label="base")
plt.plot(trend_line, label="linear trend")
plt.plot(trend_decomposed, label="seasonal_decompose trend")
plt.legend();

### 2.2 Comparing the two trends

`seasonal_decompose(period=365)` uses a centred moving average, so the first and
last ~182 days have no value. Compare only where both trends are defined.

In [ ]:
both_defined = trend_decomposed.notna()
error = mean_squared_error(trend_decomposed[both_defined], trend_line[both_defined])
print(f"points compared: {int(both_defined.sum())} of {len(series)}")
print(f"MSE between the two trend estimates: {error:.2f}")

**Question.** Repeat the comparison on a series you smoothed first. Does
smoothing change the linear trend? Why, or why not?

In [ ]:
# your code here — free exploration, not graded